# Racing Line Ground Truth Generator — Phase 1
**Purpose**: Generate optimal racing line (alpha) for any circuit from raw GPS centerline

---

| Step | Description |
|------|-------------|
| 0 | Setup |
| 1 | Upload CSV |
| 2 | Resample to 800 equal-spaced points |
| 3 | Smooth centerline |
| 4 | Generate track walls |
| 5 | Compute racing line (Minimum Curvature Path) |
| 6 | Export ground truth CSV |

**Input** : CSV file with `t_lon`, `t_lat` columns (raw centerline GPS)

**Output**: CSV with `t_lon`, `t_lat`, `alpha` — ready for Phase 2 training

> Change `TRACK_NAME` in Step 0 for each new circuit.

---
## 0. Setup

In [ ]:
!pip install scipy numpy pandas matplotlib -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.interpolate import interp1d
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'#0f0f0f','axes.facecolor':'#1a1a1a',
    'axes.edgecolor':'#444','axes.labelcolor':'#ccc',
    'xtick.color':'#888','ytick.color':'#888',
    'text.color':'#eee','grid.color':'#2a2a2a',
    'grid.linestyle':'--','figure.dpi':110,
})

# ── Change these for each circuit ─────────────────────────────────────────
TRACK_NAME  = 'Circuit_Name'   # used as output filename
TRACK_WIDTH = 9.0              # track width in metres
N_POINTS    = 800              # resampling target

# ── Coordinate conversion (JIEC latitude reference) ───────────────────────
DEG_TO_M_LAT = 111_320.0

def deg_to_m_lon(lat_ref):
    return 111_320.0 * np.cos(np.radians(lat_ref))

def plot_closed(lon_arr, lat_arr):
    return np.append(lon_arr, lon_arr[0]), np.append(lat_arr, lat_arr[0])

print('Setup complete.')

---
## 1. Upload CSV

In [ ]:
from google.colab import files

uploaded = files.upload()
CSV_FILE = list(uploaded.keys())[0]

df = pd.read_csv(CSV_FILE)

# Auto-detect column names
col_map = {}
for c in df.columns:
    cl = c.lower()
    if 'lat' in cl and 'lon' not in cl: col_map[c] = 't_lat'
    if 'lon' in cl:                     col_map[c] = 't_lon'
df.rename(columns=col_map, inplace=True)

lon = df['t_lon'].values.astype(float)
lat = df['t_lat'].values.astype(float)

lat_ref   = float(np.mean(lat))
DTM_LON   = deg_to_m_lon(lat_ref)

gap = np.sqrt(((lon[-1]-lon[0])*DTM_LON)**2 +
              ((lat[-1]-lat[0])*DEG_TO_M_LAT)**2)

print(f'File    : {CSV_FILE}')
print(f'Points  : {len(df)}')
print(f'Lat ref : {lat_ref:.4f}°')
print(f'Gap     : {gap:.1f} m  ({"closed loop ✅" if gap < 50 else "⚠️ check if closed"})')

---
## 2. Resample to Equal Spacing

In [ ]:
def resample_equal_spacing(lon_arr, lat_arr, n_points, dtm_lon):
    # Remove duplicate endpoint if present
    gap = np.sqrt(((lon_arr[-1]-lon_arr[0])*dtm_lon)**2 +
                  ((lat_arr[-1]-lat_arr[0])*DEG_TO_M_LAT)**2)
    if gap < 1.0:
        lon_arr = lon_arr[:-1]; lat_arr = lat_arr[:-1]

    dx    = np.diff(lon_arr) * dtm_lon
    dy    = np.diff(lat_arr) * DEG_TO_M_LAT
    arc   = np.concatenate([[0], np.cumsum(np.sqrt(dx**2 + dy**2))])
    total = arc[-1]

    arc_c = np.append(arc, total + np.sqrt(dx[-1]**2 + dy[-1]**2))
    f_lon = interp1d(arc_c, np.append(lon_arr, lon_arr[0]), kind='linear')
    f_lat = interp1d(arc_c, np.append(lat_arr, lat_arr[0]), kind='linear')
    s     = np.linspace(0, total, n_points, endpoint=False)

    return f_lon(s), f_lat(s), total


lon, lat, track_len = resample_equal_spacing(lon, lat, N_POINTS, DTM_LON)
spacing = track_len / N_POINTS

print(f'Track length : {track_len:.1f} m  ({track_len/1000:.2f} km)')
print(f'Points       : {len(lon)}')
print(f'Spacing      : {spacing:.3f} m per point')

---
## 3. Smooth, Check Winding & Generate Walls

In [ ]:
# ── Circular Savitzky-Golay smoothing ─────────────────────────────────────
def smooth_circular(lon_arr, lat_arr, window=15, poly=3):
    if window % 2 == 0: window += 1
    pad   = window * 3
    lp    = np.concatenate([lon_arr[-pad:], lon_arr, lon_arr[:pad]])
    lap   = np.concatenate([lat_arr[-pad:], lat_arr, lat_arr[:pad]])
    s_lon = savgol_filter(lp,  window, poly)[pad:-pad]
    s_lat = savgol_filter(lap, window, poly)[pad:-pad]
    return s_lon, s_lat

s_lon, s_lat = smooth_circular(lon, lat)

# ── Winding direction — ensure clockwise ──────────────────────────────────
area = np.sum(s_lon * DEG_TO_M_LAT * np.roll(s_lat * DEG_TO_M_LAT, -1) -
              np.roll(s_lon * DTM_LON, -1) * s_lat * DEG_TO_M_LAT) / 2.0
if area > 0:
    s_lon = s_lon[::-1].copy()
    s_lat = s_lat[::-1].copy()
    print('Winding: CCW → reversed to CW')
else:
    print('Winding: already CW ✅')

# ── Perpendicular normals (inward-pointing) ────────────────────────────────
n      = len(s_lon)
normals = np.zeros((n, 2))
for i in range(n):
    dx = (s_lon[(i+1)%n] - s_lon[(i-1)%n]) * DTM_LON
    dy = (s_lat[(i+1)%n] - s_lat[(i-1)%n]) * DEG_TO_M_LAT
    nm = np.sqrt(dx**2 + dy**2)
    if nm < 1e-10:
        normals[i] = normals[(i-1)%n]
    else:
        tang = np.array([dx/nm, dy/nm])
        normals[i] = np.array([-tang[1], tang[0]])

cx, cy = np.mean(s_lon), np.mean(s_lat)
for i in range(n):
    to_c = np.array([(cx - s_lon[i]) / DTM_LON,
                     (cy - s_lat[i]) / DEG_TO_M_LAT])
    if np.dot(normals[i], to_c) < 0:
        normals[i] = -normals[i]

# ── Track walls ────────────────────────────────────────────────────────────
nl = normals[:,0] / DTM_LON
nb = normals[:,1] / DEG_TO_M_LAT
mg = np.sqrt(nl**2 + nb**2); mg = np.where(mg < 1e-15, 1, mg)
nl /= mg; nb /= mg

hw_lon = (TRACK_WIDTH / 2) / DTM_LON
hw_lat = (TRACK_WIDTH / 2) / DEG_TO_M_LAT

il = s_lon - nl * hw_lon   # inner wall longitude
ia = s_lat - nb * hw_lat   # inner wall latitude
ol = s_lon + nl * hw_lon   # outer wall longitude
oa = s_lat + nb * hw_lat   # outer wall latitude

print(f'Track walls generated ✅  (width = {TRACK_WIDTH} m)')

---
## 4. Track Map

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
ax.fill(*plot_closed(ol, oa), color='#263238', alpha=0.95)
ax.fill(*plot_closed(il, ia), color='#0d0d0d', alpha=1.0)
ax.plot(*plot_closed(ol, oa), '-', color='#fff', lw=1.2, label='Outer wall')
ax.plot(*plot_closed(il, ia), '-', color='#fff', lw=1.2, label='Inner wall')
ax.plot(*plot_closed(s_lon, s_lat), '--', color='#ffd54f',
        lw=0.8, alpha=0.5, label='Centerline')
ax.set_title(f'{TRACK_NAME}  |  {track_len:.0f} m  |  {N_POINTS} points',
             fontsize=12)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_aspect('equal'); ax.grid(True); ax.legend()
plt.tight_layout()
plt.savefig(f'{TRACK_NAME}_map.png', bbox_inches='tight')
plt.show()

---
## 5. Minimum Curvature Path — Racing Line

Optimises lateral position (alpha) at every point to minimise
total path curvature. Lower curvature = smoother path = less energy lost at corners.

**Objective**: minimise Σκ² + λ × Σ(Δα)²
**Bounds**: α ∈ [0.05, 0.95]  (stay away from walls)
**Start**: sinusoidal — explores both inner and outer wall

In [ ]:
def racing_line_coords(alpha_arr):
    x = (il + alpha_arr * (ol - il)) * DTM_LON
    y = (ia + alpha_arr * (oa - ia)) * DEG_TO_M_LAT
    return x, y


def path_curvature_energy(alpha_arr):
    x, y   = racing_line_coords(alpha_arr)
    prev_i = np.roll(np.arange(n), 1)
    next_i = np.roll(np.arange(n), -1)

    ax_, ay_ = x[prev_i], y[prev_i]
    bx_, by_ = x,         y
    cx_, cy_ = x[next_i], y[next_i]

    ab   = np.sqrt((bx_-ax_)**2 + (by_-ay_)**2)
    bc   = np.sqrt((cx_-bx_)**2 + (cy_-by_)**2)
    ac   = np.sqrt((cx_-ax_)**2 + (cy_-ay_)**2)
    area = np.abs((bx_-ax_)*(cy_-ay_) - (cx_-ax_)*(by_-ay_)) / 2

    denom = ab * bc * ac
    kappa = np.where(denom > 1e-12, (4*area)/denom, 0.0)

    lam = 0.05
    return np.sum(kappa**2) + lam * np.sum(np.diff(alpha_arr)**2)


# Sinusoidal starting point — explores both sides of track
a0  = 0.5 + 0.3 * np.sin(np.linspace(0, 2*np.pi, n))
a0  = np.clip(a0, 0.05, 0.95)
bds = [(0.05, 0.95)] * n

print('Running MCP optimisation...')
E0 = path_curvature_energy(a0)

result = minimize(
    path_curvature_energy, a0,
    method  = 'L-BFGS-B',
    bounds  = bds,
    options = {'maxiter': 2000, 'ftol': 1e-15, 'gtol': 1e-8}
)

best_alpha = result.x
Ef         = result.fun
improvement = (E0 - Ef) / E0 * 100

print(f'Done.')
print(f'  Starting energy : {E0:.6f}')
print(f'  Final energy    : {Ef:.6f}')
print(f'  Improvement     : {improvement:.2f}%')
print(f'  α range         : {best_alpha.min():.3f} → {best_alpha.max():.3f}')
print(f'  α std           : {best_alpha.std():.3f}')

In [ ]:
# ── Racing line coordinates ────────────────────────────────────────────────
rl_lon = np.array([il[i] + best_alpha[i]*(ol[i]-il[i]) for i in range(n)])
rl_lat = np.array([ia[i] + best_alpha[i]*(oa[i]-ia[i]) for i in range(n)])

# ── Plot racing line ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Track map with racing line
axes[0].fill(*plot_closed(ol, oa), color='#1c2833', alpha=0.95)
axes[0].fill(*plot_closed(il, ia), color='#0a0a0a', alpha=1.0)
axes[0].plot(*plot_closed(ol, oa), '-', color='#fff', lw=1.0)
axes[0].plot(*plot_closed(il, ia), '-', color='#fff', lw=1.0)
axes[0].plot(*plot_closed(s_lon, s_lat), '--',
             color='#546e7a', lw=0.6, alpha=0.4, label='Centerline')
sc = axes[0].scatter(rl_lon, rl_lat, c=best_alpha,
                     cmap='RdYlGn', s=5, vmin=0, vmax=1, zorder=4)
plt.colorbar(sc, ax=axes[0], label='α (0=inner, 1=outer)')
axes[0].set_title('Racing line (green=inner, red=outer)')
axes[0].set_aspect('equal')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
axes[0].legend(fontsize=8); axes[0].grid(True)

# Alpha profile
axes[1].plot(best_alpha, color='#ff7043', lw=0.9)
axes[1].axhline(0.5, color='#ffd54f', lw=1.0,
                ls='--', alpha=0.6, label='Centerline α=0.5')
axes[1].fill_between(range(n), best_alpha, 0.5,
                     where=(best_alpha < 0.5),
                     alpha=0.3, color='#66bb6a', label='Inner')
axes[1].fill_between(range(n), best_alpha, 0.5,
                     where=(best_alpha > 0.5),
                     alpha=0.3, color='#ef5350', label='Outer')
axes[1].set_title('Lateral offset α along track')
axes[1].set_xlabel('Point index'); axes[1].set_ylabel('α')
axes[1].legend(fontsize=8); axes[1].grid(True)

plt.suptitle(f'{TRACK_NAME} — Minimum Curvature Racing Line',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(f'{TRACK_NAME}_racing_line.png', bbox_inches='tight')
plt.show()

---
## 6. Export Ground Truth CSV

In [ ]:
df_out = pd.DataFrame({
    't_lon' : s_lon,
    't_lat' : s_lat,
    'alpha' : best_alpha,
})

out_file = f'{TRACK_NAME}_ground_truth.csv'
df_out.to_csv(out_file, index=False)

print(f'Exported : {out_file}')
print(f'Shape    : {df_out.shape}  (800 rows × 3 columns)')
print(f'Columns  : t_lon, t_lat, alpha')
print(f'α range  : {best_alpha.min():.3f} → {best_alpha.max():.3f}')
print(f'α std    : {best_alpha.std():.3f}')
print()
print('Ready for Phase 2 training ✅')
df_out.head(10)

In [ ]:
from google.colab import files
import glob

for f in [out_file] + glob.glob(f'{TRACK_NAME}_*.png'):
    print(f'Downloading {f}...')
    files.download(f)